# Assignment 1 — Truck Loading and Delivery Planning

A reproducible state-space search model with greedy and A* planning, validation, and experiments.

## Problem formulation

Packages arrive in a known sequence. Each has an integer destination and each truck has a fixed capacity.
A truck is dispatched after its last assigned package arrives. To make the physical loading rule explicit,
the model treats the front-to-back unloading order as increasing destination order. The package's delivery
time is its waiting time until dispatch plus its destination distance (one time unit per destination).

The objective is a weighted sum of the number of trucks and mean package delivery time. The weight is
a modelling choice and is exposed as a parameter. This is a finite offline planning model; later cells
also discuss what changes in an online dispatch centre.

In [ ]:
from dataclasses import dataclass
from collections import deque
from heapq import heappush, heappop
from itertools import count
import random
import time
import math
import statistics

@dataclass(frozen=True)
class Package:
    arrival: int
    destination: int

def generate_problem(n, seed=0, max_destination=8):
    rng = random.Random(seed)
    return tuple(Package(i, rng.randint(1, max_destination)) for i in range(n))

def canonical_bins(bins):
    return tuple(sorted((tuple(b) for b in bins), key=lambda b: (len(b), b)))

def evaluate_plan(packages, bins, truck_weight=10.0):
    if not packages:
        return {"trucks": 0, "mean_delivery_time": 0.0, "objective": 0.0, "routes": tuple()}
    routes = []
    delivery_times = []
    for bin_ids in bins:
        dispatch_time = max(packages[i].arrival for i in bin_ids)
        route = tuple(sorted(bin_ids, key=lambda i: (packages[i].destination, packages[i].arrival)))
        routes.append(route)
        for i in route:
            delivery_times.append(dispatch_time - packages[i].arrival + packages[i].destination)
    mean_time = sum(delivery_times) / len(packages)
    objective = truck_weight * len(bins) + mean_time
    return {"trucks": len(bins), "mean_delivery_time": mean_time,
            "objective": objective, "routes": tuple(routes)}

def valid_plan(packages, bins, capacity):
    flat = [i for b in bins for i in b]
    return (len(flat) == len(packages) and sorted(flat) == list(range(len(packages)))
            and all(0 < len(b) <= capacity for b in bins))

## Search algorithms

In [ ]:
def successor_states(packages, bins, next_index, capacity):
    package_id = next_index
    for j, truck in enumerate(bins):
        if len(truck) < capacity:
            updated = list(bins)
            updated[j] = truck + (package_id,)
            yield canonical_bins(updated)
    if len(bins) < len(packages):
        yield canonical_bins(list(bins) + [(package_id,)])

def lower_bound(packages, bins, next_index, capacity, truck_weight):
    remaining = len(packages) - next_index
    free_slots = sum(capacity - len(truck) for truck in bins)
    minimum_extra = math.ceil(max(0, remaining - free_slots) / capacity)
    truck_lb = len(bins) + minimum_extra
    if not packages:
        return 0.0
    unavoidable_travel = sum(p.destination for p in packages) / len(packages)
    return truck_weight * truck_lb + unavoidable_travel

def astar_plan(packages, capacity=3, truck_weight=10.0, max_expansions=500000):
    packages = tuple(packages)
    serial = count()
    start = (0, tuple())
    heap = [(lower_bound(packages, tuple(), 0, capacity, truck_weight), next(serial), start)]
    best_seen = {}
    expanded = 0
    best_result = None
    while heap and expanded < max_expansions:
        _, _, (idx, bins) = heappop(heap)
        key = (idx, bins)
        if key in best_seen:
            continue
        best_seen[key] = True
        expanded += 1
        if idx == len(packages):
            result = evaluate_plan(packages, bins, truck_weight)
            if best_result is None or result["objective"] < best_result["objective"]:
                best_result = result | {"bins": bins}
            continue
        for child in successor_states(packages, bins, idx, capacity):
            child_state = (idx + 1, child)
            priority = lower_bound(packages, child, idx + 1, capacity, truck_weight)
            heappush(heap, (priority, next(serial), child_state))
    return best_result, expanded

def greedy_plan(packages, capacity=3, truck_weight=10.0):
    bins = []
    for i, package in enumerate(packages):
        choices = []
        for j, truck in enumerate(bins):
            if len(truck) < capacity:
                candidate = list(bins)
                candidate[j] = truck + (i,)
                candidate = canonical_bins(candidate)
                choices.append((evaluate_plan(packages[:i+1], candidate, truck_weight)["objective"], candidate))
        if len(bins) < len(packages):
            candidate = canonical_bins(list(bins) + [(i,)])
            choices.append((evaluate_plan(packages[:i+1], candidate, truck_weight)["objective"], candidate))
        bins = min(choices, key=lambda x: x[0])[1]
    result = evaluate_plan(packages, bins, truck_weight)
    return result | {"bins": bins}

def breadth_first_plan(packages, capacity=3, truck_weight=10.0, max_expansions=500000):
    packages = tuple(packages)
    queue = deque([(0, tuple())])
    expanded = 0
    best = None
    while queue and expanded < max_expansions:
        idx, bins = queue.popleft()
        expanded += 1
        if idx == len(packages):
            result = evaluate_plan(packages, bins, truck_weight)
            if best is None or result["objective"] < best["objective"]:
                best = result | {"bins": bins}
            continue
        for child in successor_states(packages, bins, idx, capacity):
            queue.append((idx + 1, child))
    return best, expanded

## Run a small instance and validate the result

In [ ]:
packages = generate_problem(8, seed=11, max_destination=7)
capacity = 3
for i, package in enumerate(packages):
    print(i, package)

plan, expanded = astar_plan(packages, capacity=capacity, truck_weight=10.0)
assert plan is not None
assert valid_plan(packages, plan["bins"], capacity)
print("\nA* plan:", plan)
print("Expanded states:", expanded)

## Compare algorithms across generated instances

In [ ]:
def benchmark_truck_algorithms(problem_sizes=(5, 7, 9), seeds=range(5), capacity=3):
    rows = []
    for n in problem_sizes:
        for seed in seeds:
            packages = generate_problem(n, seed=seed)
            for name, solver in (
                ("Greedy", lambda: greedy_plan(packages, capacity)),
                ("BFS", lambda: breadth_first_plan(packages, capacity, max_expansions=200000)),
                ("A*", lambda: astar_plan(packages, capacity, max_expansions=200000))
            ):
                start = time.perf_counter()
                result = solver()
                elapsed = time.perf_counter() - start
                if name in ("A*", "BFS"):
                    solution, nodes = result
                else:
                    solution = result
                    nodes = None
                rows.append({"n": n, "seed": seed, "algorithm": name,
                             "objective": solution["objective"] if solution else None,
                             "trucks": solution["trucks"] if solution else None,
                             "nodes_expanded": nodes, "seconds": elapsed})
    return rows

truck_results = benchmark_truck_algorithms()
for row in truck_results:
    print(row)

## Sensitivity to the truck-count weight

In [ ]:
def weight_sensitivity(packages, weights=(0.0, 2.0, 5.0, 10.0, 20.0), capacity=3):
    results = []
    for weight in weights:
        plan, nodes = astar_plan(packages, capacity, truck_weight=weight)
        results.append({"truck_weight": weight, "trucks": plan["trucks"],
                        "mean_delivery_time": plan["mean_delivery_time"],
                        "objective": plan["objective"], "nodes": nodes})
    return results

weight_sensitivity(packages)

## Interpretation and modelling limitations

- A larger truck-count weight makes consolidation more valuable, even when some packages wait longer.
- A lower weight favours faster delivery and may justify more trucks.
- The A* lower bound is deliberately optimistic: it counts the minimum number of additional trucks and
  unavoidable destination travel. It is useful for ordering search but does not make the full problem cheap.
- The model assumes all packages are known before the plan is chosen, one truck capacity, equal travel-time
  units per destination, and no route congestion or loading-time differences.
- A real online system would need a dispatch policy, uncertain arrivals, truck availability, and possibly
  a rolling-horizon planner. Those are variations, not facts specified by the assignment.